In [ ]:
%pip install -q xgboost shap openpyxl hmmlearn

In [ ]:
import os
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import xgboost as xgb
import shap

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    RocCurveDisplay, PrecisionRecallDisplay, ConfusionMatrixDisplay,
    classification_report, silhouette_score,
    davies_bouldin_score, calinski_harabasz_score
)
from sklearn.inspection import permutation_importance
from sklearn.cluster import KMeans
from sklearn.mixture import GaussianMixture
from sklearn.decomposition import PCA

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
warnings.filterwarnings("ignore", category=FutureWarning)

sns.set_theme(style="whitegrid", context="notebook", palette="colorblind")
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "axes.titleweight": "bold",
    "axes.labelweight": "medium",
    "figure.autolayout": False
})

FIGURE_DIR = "PrediCare_Figures"
os.makedirs(FIGURE_DIR, exist_ok=True)

def save_figure(name):
    plt.savefig(os.path.join(FIGURE_DIR, name), dpi=300, bbox_inches="tight")

print("Libraries and plotting settings ready.")

In [ ]:
from google.colab import files

uploaded = files.upload()

In [ ]:
FILES = {
    4: "4 years_datas.xlsx",
    5: "5 years_datas.xlsx",
    6: "6 years_datas.xlsx",
    7: "7 years_datas.xlsx",
    8: "8 years_datas.xlsx",
}

frames = []
for years, path in FILES.items():
    if not os.path.exists(path):
        raise FileNotFoundError(f"Missing dataset file: {path}")
    frame = pd.read_excel(path)
    frame["followup_years"] = years
    frames.append(frame)

raw = pd.concat(frames, ignore_index=True)

required_columns = {
    "patientid", "timed", "type_3", "age", "sex",
    "weight", "height", "glu", "HbA1c", "SBP", "DBP", "rate"
}
missing_columns = required_columns.difference(raw.columns)
if missing_columns:
    raise ValueError(f"Missing required dataset columns: {sorted(missing_columns)}")

print("Combined dataset shape:", raw.shape)
print("Unique patient IDs:", raw["patientid"].nunique())
display(raw.head())

In [ ]:
status_check = raw.dropna(subset=["type_1", "type_2", "type_3", "type_1_2"])
print("type_3 matches max(type_1, type_2):",
      (status_check["type_3"] == status_check[["type_1", "type_2"]].max(axis=1)).mean().round(4))
print("type_1_2 matches type_1 + type_2:",
      (status_check["type_1_2"] == status_check[["type_1", "type_2"]].sum(axis=1)).mean().round(4))

In [ ]:
baseline = raw.loc[raw["timed"] == 1].copy()
followup = raw.loc[raw["timed"] == 2, ["patientid", "type_3"]].rename(
    columns={"type_3": "type_3_followup"}
)

if baseline["patientid"].duplicated().any() or followup["patientid"].duplicated().any():
    raise ValueError(
        "Duplicate patient IDs found in baseline or follow-up records. "
        "Resolve duplicate patient records before merging to avoid many-to-many matches."
    )

cohort = baseline.merge(
    followup, on="patientid", how="inner", validate="one_to_one"
)
cohort = cohort.loc[cohort["type_3"] == 1].copy()
cohort["progressed_to_t2d"] = (cohort["type_3_followup"] == 2).astype(int)

numeric_columns = ["age", "weight", "height", "glu", "HbA1c", "rate", "SBP", "DBP"]
for column in numeric_columns:
    cohort[column] = pd.to_numeric(cohort[column], errors="coerce")

cohort.loc[~cohort["height"].between(100, 230), "height"] = np.nan
cohort.loc[~cohort["weight"].between(20, 350), "weight"] = np.nan
cohort["bmi"] = cohort["weight"] / (cohort["height"] / 100) ** 2
cohort.loc[~cohort["bmi"].between(10, 80), "bmi"] = np.nan
cohort["pulse_pressure"] = cohort["SBP"] - cohort["DBP"]
cohort["sex"] = pd.to_numeric(cohort["sex"], errors="coerce")

FEATURE_COLS = [
    "age", "sex", "bmi", "glu", "HbA1c",
    "SBP", "DBP", "pulse_pressure", "rate"
]
df = cohort[
    ["patientid", "followup_years"] + FEATURE_COLS + ["progressed_to_t2d"]
].reset_index(drop=True)

if df.empty or df["progressed_to_t2d"].nunique() != 2:
    raise ValueError("The eligible cohort must contain both progression classes.")

print("Eligible baseline-prediabetes cohort:", df.shape)
print("Progression counts:")
display(df["progressed_to_t2d"].value_counts().rename(index={0: "No progression", 1: "Progressed"}))
print("Invalid BMI values set to missing:", int(df["bmi"].isna().sum()))
display(df.head())

In [ ]:
df[FEATURE_COLS + ['progressed_to_t2d']].describe().T


In [ ]:
print('Missing values (%):')
print((df[FEATURE_COLS].isna().mean() * 100).round(1).sort_values(ascending=False))


In [ ]:
class_counts = df["progressed_to_t2d"].value_counts().reindex([0, 1], fill_value=0)
labels = ["No progression", "Progressed to T2D"]

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(labels, class_counts.values, color=["#4C78A8", "#E45756"], width=0.6)
ax.bar_label(bars, labels=[f"{v:,}\n{v / class_counts.sum():.1%}" for v in class_counts.values],
             padding=4, fontsize=10)
ax.set_title("Progression outcome distribution")
ax.set_ylabel("Number of patients")
ax.set_ylim(0, max(class_counts.values) * 1.18)
save_figure("01_outcome_distribution.png")
plt.show()

In [ ]:
numeric_features = ["age", "bmi", "glu", "HbA1c", "SBP", "DBP", "rate"]
fig, axes = plt.subplots(3, 3, figsize=(13, 10))
axes = axes.flatten()

for index, feature in enumerate(numeric_features):
    sns.histplot(
        data=df, x=feature, hue="progressed_to_t2d",
        stat="density", common_norm=False, kde=True,
        element="step", fill=False, ax=axes[index]
    )
    axes[index].set_title(feature)
    axes[index].set_xlabel(feature)

axes[-1].axis("off")
fig.suptitle("Baseline feature distributions by progression outcome", y=1.02, fontsize=14)
plt.tight_layout()
save_figure("02_feature_distributions.png")
plt.show()

In [ ]:
correlation = df[numeric_features + ["sex", "progressed_to_t2d"]].corr()
fig, ax = plt.subplots(figsize=(9, 7))
sns.heatmap(correlation, annot=True, fmt=".2f", cmap="vlag",
            center=0, square=True, linewidths=0.5, ax=ax)
ax.set_title("Correlation matrix of baseline features and outcome")
plt.tight_layout()
save_figure("03_correlation_heatmap.png")
plt.show()

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(14, 7))
axes = axes.flatten()
for index, feature in enumerate(numeric_features + ["pulse_pressure"]):
    sns.boxplot(data=df, y=feature, color="#72B7B2", ax=axes[index])
    axes[index].set_title(feature)
    axes[index].set_xlabel("")
fig.suptitle("Baseline feature spread and potential outliers", y=1.02, fontsize=14)
plt.tight_layout()
save_figure("04_feature_boxplots.png")
plt.show()

In [ ]:
print("Progression rate by sex (%):")
display((pd.crosstab(df["sex"], df["progressed_to_t2d"], normalize="index") * 100).round(1))
print("Progression rate by follow-up duration (%):")
display((pd.crosstab(df["followup_years"], df["progressed_to_t2d"], normalize="index") * 100).round(1))

duration_rates = (
    df.groupby("followup_years")["progressed_to_t2d"].mean().mul(100).reset_index()
)
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(data=duration_rates, x="followup_years", y="progressed_to_t2d",
            color="#4C78A8", ax=ax)
ax.set_title("Observed progression proportion by follow-up duration")
ax.set_xlabel("Follow-up duration (years)")
ax.set_ylabel("Progressed to T2D (%)")
ax.set_ylim(0, 100)
save_figure("05_progression_by_followup.png")
plt.show()

In [ ]:
if df["patientid"].duplicated().any():
    raise ValueError("Patient IDs are duplicated. Use a patient-grouped split before model evaluation.")

X = df[FEATURE_COLS].copy()
y = df["progressed_to_t2d"].astype(int).copy()

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, stratify=y, random_state=RANDOM_STATE
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=RANDOM_STATE
)

X_trainval = pd.concat([X_train, X_val])
y_trainval = pd.concat([y_train, y_val])

preprocess = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), FEATURE_COLS)
])

print(f"Train: {len(X_train)} | Validation: {len(X_val)} | Test: {len(X_test)}")

In [ ]:
negative_count, positive_count = np.bincount(y_trainval)
positive_weight = negative_count / positive_count

models = {
    "Logistic Regression": LogisticRegression(
        max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=400, max_depth=6, min_samples_leaf=5,
        class_weight="balanced", random_state=RANDOM_STATE, n_jobs=-1
    ),
    "XGBoost": xgb.XGBClassifier(
        n_estimators=300, max_depth=3, learning_rate=0.05,
        scale_pos_weight=positive_weight, eval_metric="logloss",
        random_state=RANDOM_STATE, n_jobs=-1
    ),
    "SVM (RBF)": SVC(
        kernel="rbf", C=1.0, gamma="scale", probability=True,
        class_weight="balanced", random_state=RANDOM_STATE
    )
}

print(f"Class imbalance weight (negative/positive): {positive_weight:.2f}")

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scoring = ["accuracy", "precision", "recall", "f1", "roc_auc", "average_precision"]

cv_rows, test_rows = [], []
fitted_pipelines, roc_data = {}, {}

for name, estimator in models.items():
    pipeline = Pipeline([("preprocess", preprocess), ("model", estimator)])
    cv_results = cross_validate(
        pipeline, X_trainval, y_trainval, cv=cv,
        scoring=scoring, n_jobs=-1, error_score="raise"
    )

    cv_rows.append({
        "Model": name,
        "CV Accuracy": cv_results["test_accuracy"].mean(),
        "CV Precision": cv_results["test_precision"].mean(),
        "CV Recall": cv_results["test_recall"].mean(),
        "CV F1": cv_results["test_f1"].mean(),
        "CV ROC-AUC": cv_results["test_roc_auc"].mean(),
        "CV ROC-AUC std": cv_results["test_roc_auc"].std(),
        "CV PR-AUC": cv_results["test_average_precision"].mean()
    })

    pipeline.fit(X_trainval, y_trainval)
    fitted_pipelines[name] = pipeline

    test_predictions = pipeline.predict(X_test)
    test_probabilities = pipeline.predict_proba(X_test)[:, 1]
    tn, fp, fn, tp = confusion_matrix(y_test, test_predictions, labels=[0, 1]).ravel()

    test_rows.append({
        "Model": name,
        "Test Accuracy": accuracy_score(y_test, test_predictions),
        "Test Precision": precision_score(y_test, test_predictions, zero_division=0),
        "Test Recall": recall_score(y_test, test_predictions, zero_division=0),
        "Test F1": f1_score(y_test, test_predictions, zero_division=0),
        "Test ROC-AUC": roc_auc_score(y_test, test_probabilities),
        "Test PR-AUC": average_precision_score(y_test, test_probabilities),
        "Test Specificity": tn / max(1, tn + fp)
    })
    roc_data[name] = (y_test.copy(), test_probabilities)

cv_df = pd.DataFrame(cv_rows).sort_values("CV ROC-AUC", ascending=False)
test_df = pd.DataFrame(test_rows).sort_values("Test ROC-AUC", ascending=False)

print("Five-fold cross-validation results:")
display(cv_df.round(3))
print("Held-out test results:")
display(test_df.round(3))

In [ ]:
for name, pipeline in fitted_pipelines.items():
    predictions = pipeline.predict(X_test)
    print(f"\n{name}")
    print(classification_report(
        y_test, predictions, target_names=["No progression", "Progressed"],
        zero_division=0
    ))

    ConfusionMatrixDisplay.from_predictions(
        y_test, predictions,
        display_labels=["No progression", "Progressed"],
        cmap="Blues", values_format="d"
    )
    plt.title(f"Confusion matrix – {name}")
    plt.tight_layout()
    save_figure(f"confusion_matrix_{name.lower().replace(' ', '_').replace('(', '').replace(')', '')}.png")
    plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 6))
for name, (actual, probabilities) in roc_data.items():
    RocCurveDisplay.from_predictions(actual, probabilities, name=name, ax=ax)
ax.plot([0, 1], [0, 1], linestyle="--", color="grey", label="Random baseline")
ax.set_title("ROC curve comparison – four progression classifiers")
plt.tight_layout()
save_figure("06_classifier_roc_curves.png")
plt.show()

In [ ]:
comparison_metrics = [
    "Test Accuracy", "Test Precision", "Test Recall",
    "Test F1", "Test ROC-AUC", "Test PR-AUC", "Test Specificity"
]
comparison_long = test_df.melt(
    id_vars="Model", value_vars=comparison_metrics,
    var_name="Metric", value_name="Score"
)
fig, ax = plt.subplots(figsize=(12, 6))
sns.barplot(data=comparison_long, x="Metric", y="Score", hue="Model", ax=ax)
ax.set_ylim(0, 1)
ax.set_title("Held-out test performance comparison – four classifiers")
ax.set_ylabel("Score")
ax.set_xlabel("")
plt.xticks(rotation=20, ha="right")
ax.legend(title="Classifier", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
save_figure("07_classifier_metric_comparison.png")
plt.show()

In [ ]:
tree_candidates = [name for name in ["XGBoost", "Random Forest"] if name in fitted_pipelines]
cv_lookup = cv_df.set_index("Model")
best_tree_name = max(tree_candidates, key=lambda name: cv_lookup.loc[name, "CV ROC-AUC"])
best_pipe = fitted_pipelines[best_tree_name]
print("Tree model selected for SHAP:", best_tree_name)

X_test_transformed = best_pipe.named_steps["preprocess"].transform(X_test)
X_test_df = pd.DataFrame(X_test_transformed, columns=FEATURE_COLS, index=X_test.index)

explainer = shap.TreeExplainer(best_pipe.named_steps["model"])
shap_values = explainer.shap_values(X_test_df)

In [ ]:
shap.summary_plot(shap_values, X_test_df, show=False)
plt.title(f"SHAP summary – {best_tree_name}")
plt.tight_layout()
save_figure("08_shap_summary.png")
plt.show()

In [ ]:
shap.summary_plot(shap_values, X_test_df, plot_type="bar", show=False)
plt.title(f"Mean absolute SHAP importance – {best_tree_name}")
plt.tight_layout()
save_figure("09_shap_feature_importance.png")
plt.show()

In [ ]:
explainer = shap.TreeExplainer(best_pipe.named_steps["model"])
raw_shap = explainer.shap_values(X_test_df)

if isinstance(raw_shap, list):
    shap_values = raw_shap[1]
    base_value = explainer.expected_value[1]
elif np.ndim(raw_shap) == 3:
    shap_values = raw_shap[:, :, 1]
    base_value = np.asarray(explainer.expected_value).reshape(-1)[-1]
else:
    shap_values = raw_shap
    base_value = np.asarray(explainer.expected_value).reshape(-1)[-1]

if np.ndim(shap_values) != 2:
    raise ValueError("Unexpected SHAP output shape; inspect the installed SHAP version.")

In [ ]:
patient_index = 0
shap.force_plot(
    base_value, shap_values[patient_index],
    X_test_df.iloc[patient_index], matplotlib=True, show=False
)
plt.title("Local explanation for one held-out patient")
plt.tight_layout()
save_figure("10_shap_local_explanation.png")
plt.show()
print("Actual outcome:", "Progressed to T2D" if y_test.iloc[patient_index] == 1 else "No progression")

In [ ]:
importance = permutation_importance(
    best_pipe, X_test, y_test, n_repeats=20,
    random_state=RANDOM_STATE, scoring="roc_auc", n_jobs=-1
)
importance_df = pd.DataFrame({
    "Feature": FEATURE_COLS,
    "Importance mean": importance.importances_mean,
    "Importance std": importance.importances_std
}).sort_values("Importance mean", ascending=True)

display(importance_df.sort_values("Importance mean", ascending=False).round(4))
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(importance_df["Feature"], importance_df["Importance mean"],
        xerr=importance_df["Importance std"], color="#4C78A8", capsize=3)
ax.set_xlabel("Decrease in ROC-AUC after feature permutation")
ax.set_title(f"Permutation feature importance – {best_tree_name}")
plt.tight_layout()
save_figure("11_permutation_feature_importance.png")
plt.show()

In [ ]:
best_classifier_name = cv_df.iloc[0]["Model"]
best_classifier = fitted_pipelines[best_classifier_name]
best_probabilities = best_classifier.predict_proba(X_test)[:, 1]

def assign_risk_band(probability):
    if probability < 0.33:
        return "Low"
    if probability < 0.66:
        return "Moderate"
    return "High"

risk_table = X_test.copy()
risk_table["Actual outcome"] = y_test.values
risk_table["Predicted probability"] = best_probabilities.round(3)
risk_table["Exploratory risk band"] = [
    assign_risk_band(value) for value in best_probabilities
]
print("Best classifier by cross-validation ROC-AUC:", best_classifier_name)
display(risk_table.head(10))

## K-Means and Gaussian Mixture Model

These models explore patient subgroups using baseline measurements. Cluster labels are statistical groups, not clinical diagnoses.

In [ ]:
cluster_preprocessor = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])
X_cluster = cluster_preprocessor.fit_transform(df[FEATURE_COLS])

# Compare candidate cluster counts using silhouette (higher is better).
cluster_rows = []
kmeans_models = {}
gmm_models = {}
for k in range(2, 7):
    km = KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE)
    km_labels = km.fit_predict(X_cluster)
    km_sil = silhouette_score(X_cluster, km_labels, sample_size=min(3000, len(X_cluster)), random_state=RANDOM_STATE)
    kmeans_models[k] = (km, km_labels)

    gm = GaussianMixture(n_components=k, covariance_type='diag', n_init=3,
                         max_iter=300, random_state=RANDOM_STATE, reg_covar=1e-4)
    gm_labels = gm.fit_predict(X_cluster)
    gm_sil = silhouette_score(X_cluster, gm_labels, sample_size=min(3000, len(X_cluster)), random_state=RANDOM_STATE)
    gmm_models[k] = (gm, gm_labels)

    cluster_rows.extend([
        {'Model': 'K-Means', 'Clusters': k, 'Silhouette': km_sil,
         'Davies-Bouldin': davies_bouldin_score(X_cluster, km_labels),
         'Calinski-Harabasz': calinski_harabasz_score(X_cluster, km_labels),
         'AIC': np.nan, 'BIC': np.nan},
        {'Model': 'GMM', 'Clusters': k, 'Silhouette': gm_sil,
         'Davies-Bouldin': davies_bouldin_score(X_cluster, gm_labels),
         'Calinski-Harabasz': calinski_harabasz_score(X_cluster, gm_labels),
         'AIC': gm.aic(X_cluster), 'BIC': gm.bic(X_cluster)}
    ])

cluster_comparison = pd.DataFrame(cluster_rows)
display(cluster_comparison.round(3))

# Select K-Means by silhouette and GMM by BIC (lower is better).
best_k_kmeans = int(cluster_comparison.query("Model == 'K-Means'").sort_values(
    'Silhouette', ascending=False).iloc[0]['Clusters'])
best_k_gmm = int(cluster_comparison.query("Model == 'GMM'").sort_values(
    'BIC', ascending=True).iloc[0]['Clusters'])

best_kmeans, kmeans_labels = kmeans_models[best_k_kmeans]
best_gmm, gmm_labels = gmm_models[best_k_gmm]
kmeans_quality = {
    'Model': 'K-Means', 'Selected clusters': best_k_kmeans,
    'Silhouette': silhouette_score(X_cluster, kmeans_labels, sample_size=min(3000, len(X_cluster)), random_state=RANDOM_STATE),
    'Davies-Bouldin': davies_bouldin_score(X_cluster, kmeans_labels),
    'Calinski-Harabasz': calinski_harabasz_score(X_cluster, kmeans_labels),
    'AIC': np.nan, 'BIC': np.nan
}
gmm_quality = {
    'Model': 'GMM', 'Selected clusters': best_k_gmm,
    'Silhouette': silhouette_score(X_cluster, gmm_labels, sample_size=min(3000, len(X_cluster)), random_state=RANDOM_STATE),
    'Davies-Bouldin': davies_bouldin_score(X_cluster, gmm_labels),
    'Calinski-Harabasz': calinski_harabasz_score(X_cluster, gmm_labels),
    'AIC': best_gmm.aic(X_cluster), 'BIC': best_gmm.bic(X_cluster)
}
cluster_quality_summary = pd.DataFrame([kmeans_quality, gmm_quality])
print(f"Selected K-Means: k={best_k_kmeans} (highest silhouette)")
print(f"Selected GMM: components={best_k_gmm} (lowest BIC)")

# Compare silhouette across methods and cluster counts.
fig, ax = plt.subplots(figsize=(9, 5))
sns.lineplot(data=cluster_comparison, x='Clusters', y='Silhouette',
             hue='Model', marker='o', ax=ax)
ax.set_title('K-Means vs GMM: silhouette comparison')
plt.tight_layout()
save_figure("12_clustering_silhouette_comparison.png")
plt.show()

# GMM information-criterion comparison.
gmm_ic = cluster_comparison.query("Model == 'GMM'").melt(
    id_vars='Clusters', value_vars=['AIC', 'BIC'],
    var_name='Criterion', value_name='Score')
fig, ax = plt.subplots(figsize=(8, 5))
sns.lineplot(data=gmm_ic, x='Clusters', y='Score', hue='Criterion',
             marker='o', ax=ax)
ax.set_title('GMM model selection using AIC and BIC (lower is better)')
plt.tight_layout()
save_figure("13_gmm_aic_bic.png")
plt.show()

pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_cluster)
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=kmeans_labels,
                palette='tab10', s=24, linewidth=0, ax=axes[0], legend='brief')
axes[0].set_title(f'K-Means patient profiles (k={best_k_kmeans})')
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=gmm_labels,
                palette='tab10', s=24, linewidth=0, ax=axes[1], legend='brief')
axes[1].set_title(f'GMM patient profiles (components={best_k_gmm})')
for ax in axes:
    ax.set_xlabel('PC1')
    ax.set_ylabel('PC2')
plt.tight_layout()
save_figure("14_pca_cluster_visualization.png")
plt.show()

# Describe the clinical profile of each cluster using original units.
cluster_profiles = df[FEATURE_COLS].copy()
cluster_profiles['KMeans_cluster'] = kmeans_labels
cluster_profiles['GMM_cluster'] = gmm_labels
cluster_profiles['progressed_to_t2d'] = y.values
print("K-Means cluster profiles:")
display(cluster_profiles.groupby('KMeans_cluster').mean(numeric_only=True).round(2))
print("GMM cluster profiles:")
display(cluster_profiles.groupby('GMM_cluster').mean(numeric_only=True).round(2))

fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.barplot(data=cluster_profiles, x='KMeans_cluster', y='progressed_to_t2d',
            errorbar=None, ax=axes[0])
axes[0].set_title('Observed progression fraction by K-Means cluster')
axes[0].set_ylabel('Fraction progressed')
sns.barplot(data=cluster_profiles, x='GMM_cluster', y='progressed_to_t2d',
            errorbar=None, ax=axes[1])
axes[1].set_title('Observed progression fraction by GMM cluster')
axes[1].set_ylabel('Fraction progressed')
plt.tight_layout()
save_figure("15_progression_by_cluster.png")
plt.show()


## Hidden Markov Model

The HMM explores two-visit sequences. With only two observations per patient, interpret its hidden states as exploratory patterns rather than confirmed disease stages.

In [ ]:
# Optional dependency: hmmlearn is installed in the first cell.
hmm_comparison = {'Model': 'HMM', 'Status': 'Not fitted',
                  'Log-likelihood per observation': np.nan,
                  'Number of sequences': 0, 'Number of observations': 0}
try:
    from hmmlearn.hmm import GaussianHMM

    hmm_features = ['age', 'bmi', 'glu', 'HbA1c', 'SBP', 'DBP', 'rate']
    seq_raw = raw.copy()

    for col in ['age', 'weight', 'height', 'glu', 'HbA1c', 'SBP', 'DBP', 'rate']:
        seq_raw[col] = pd.to_numeric(seq_raw[col], errors='coerce')
    seq_raw['bmi'] = seq_raw['weight'] / (seq_raw['height'] / 100.0) ** 2
    seq_raw['bmi'] = seq_raw['bmi'].where(seq_raw['bmi'].between(10, 80))
    seq_raw['pulse_pressure'] = seq_raw['SBP'] - seq_raw['DBP']

    # Restrict to baseline-prediabetes patients with both timed observations.
    eligible_ids = set(cohort['patientid'].dropna().unique())
    seq_raw = seq_raw[seq_raw['patientid'].isin(eligible_ids) &
                      seq_raw['timed'].isin([1, 2])].copy()
    seq_raw = seq_raw.sort_values(['patientid', 'timed'])

    sequences, lengths = [], []
    for pid, g in seq_raw.groupby('patientid'):
        g = g.drop_duplicates(subset=['timed']).sort_values('timed')
        if set(g['timed']) == {1, 2}:
            arr = g[hmm_features].to_numpy(dtype=float)
            if arr.shape == (2, len(hmm_features)):
                sequences.append(arr)
                lengths.append(2)

    if len(sequences) >= 20:
        seq_matrix = np.vstack(sequences)
        hmm_imputer = SimpleImputer(strategy='median')
        seq_matrix = hmm_imputer.fit_transform(seq_matrix)
        hmm_scaler = StandardScaler()
        seq_matrix = hmm_scaler.fit_transform(seq_matrix)

        # Fit a small-state Gaussian HMM. Multiple random starts are avoided here
        # to keep the notebook reproducible and computationally manageable.
        hmm_model = GaussianHMM(n_components=3, covariance_type='diag',
                                n_iter=200, random_state=RANDOM_STATE,
                                min_covar=1e-3)
        hmm_model.fit(seq_matrix, lengths)
        hidden_states = hmm_model.predict(seq_matrix, lengths)
        hmm_comparison = {
            'Model': 'HMM', 'Status': 'Fitted (exploratory)',
            'Log-likelihood per observation': hmm_model.score(seq_matrix, lengths) / len(seq_matrix),
            'Number of sequences': len(lengths),
            'Number of observations': len(seq_matrix)
        }

        print(f"HMM fitted on {len(sequences)} patient sequences "
              f"({len(seq_matrix)} observations).")
        print("Hidden-state transition matrix:")
        display(pd.DataFrame(hmm_model.transmat_,
                             index=[f'State {i}' for i in range(3)],
                             columns=[f'State {i}' for i in range(3)]).round(3))

        state_counts = pd.Series(hidden_states).value_counts().sort_index()
        fig, ax = plt.subplots(figsize=(6, 4))
        state_counts.plot(kind='bar', ax=ax)
        ax.set_title('HMM hidden-state observation counts')
        ax.set_xlabel('Hidden state')
        ax.set_ylabel('Number of observations')
        plt.tight_layout()
        save_figure("16_hmm_state_counts.png")
        plt.show()

        fig, ax = plt.subplots(figsize=(6, 5))
        sns.heatmap(hmm_model.transmat_, annot=True, fmt='.2f',
                    cmap='Blues', xticklabels=[f'S{i}' for i in range(3)],
                    yticklabels=[f'S{i}' for i in range(3)], ax=ax)
        ax.set_title('HMM hidden-state transition probabilities')
        ax.set_xlabel('Next hidden state')
        ax.set_ylabel('Current hidden state')
        plt.tight_layout()
        save_figure("17_hmm_transition_matrix.png")
        plt.show()
    else:
        print("Not enough complete two-visit patient sequences to fit HMM reliably.")
except ImportError:
    hmm_comparison['Status'] = 'Not fitted: hmmlearn unavailable'
    print("hmmlearn is unavailable. Run: !pip install hmmlearn, then rerun this cell.")
except Exception as e:
    hmm_comparison['Status'] = f'Not fitted: {type(e).__name__}'
    print("HMM section could not be completed:", repr(e))
    print("Check available follow-up records, missing values and hmmlearn installation.")


## Comparison of all 7 models and final model selection

The four classifiers predict progression. K-Means and GMM discover patient subgroups. HMM explores sequential states. Each model family is evaluated with suitable metrics; their scores are not combined into one ranking.

In [ ]:
classifier_comparison = cv_df.merge(test_df, on="Model", how="inner")
classifier_comparison = classifier_comparison.sort_values(
    ["CV ROC-AUC", "CV ROC-AUC std"], ascending=[False, True]
)

print("1. Four progression classifiers")
display(classifier_comparison.round(3))

fig, ax = plt.subplots(figsize=(12, 6))
long_metrics = classifier_comparison.melt(
    id_vars="Model",
    value_vars=["Test Accuracy", "Test Precision", "Test Recall", "Test F1",
                "Test ROC-AUC", "Test PR-AUC", "Test Specificity"],
    var_name="Metric", value_name="Score"
)
sns.barplot(data=long_metrics, x="Metric", y="Score", hue="Model", ax=ax)
ax.set_ylim(0, 1)
ax.set_title("Comparison of four progression classifiers")
ax.set_ylabel("Test score")
ax.set_xlabel("")
plt.xticks(rotation=20, ha="right")
ax.legend(title="Classifier", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.tight_layout()
save_figure("18_all_classifier_metrics.png")
plt.show()

fig, ax = plt.subplots(figsize=(7, 6))
for name, (actual, probabilities) in roc_data.items():
    RocCurveDisplay.from_predictions(actual, probabilities, name=name, ax=ax)
ax.plot([0, 1], [0, 1], "--", color="grey")
ax.set_title("ROC curves – progression classifiers")
plt.tight_layout()
save_figure("19_all_classifier_roc.png")
plt.show()

fig, ax = plt.subplots(figsize=(7, 6))
for name, pipeline in fitted_pipelines.items():
    PrecisionRecallDisplay.from_predictions(
        y_test, pipeline.predict_proba(X_test)[:, 1], name=name, ax=ax
    )
ax.set_title("Precision–Recall curves – progression classifiers")
plt.tight_layout()
save_figure("20_all_classifier_precision_recall.png")
plt.show()

ranked = classifier_comparison.sort_values("CV ROC-AUC")
fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(ranked["Model"], ranked["CV ROC-AUC"],
        xerr=ranked["CV ROC-AUC std"], capsize=4, color="#4C78A8")
ax.set_xlim(0, 1)
ax.set_xlabel("Mean 5-fold CV ROC-AUC")
ax.set_title("Classifier ranking for final selection")
plt.tight_layout()
save_figure("21_classifier_cv_ranking.png")
plt.show()

print("2. K-Means and GMM")
display(cluster_quality_summary.round(3))
cluster_metrics = cluster_quality_summary.melt(
    id_vars="Model",
    value_vars=["Silhouette", "Davies-Bouldin", "Calinski-Harabasz"],
    var_name="Metric", value_name="Score"
)
fig, ax = plt.subplots(figsize=(9, 5))
sns.barplot(data=cluster_metrics, x="Metric", y="Score", hue="Model", ax=ax)
ax.set_title("Clustering quality – K-Means vs GMM")
plt.tight_layout()
save_figure("22_kmeans_gmm_quality.png")
plt.show()

print("3. Hidden Markov Model")
hmm_comparison_df = pd.DataFrame([hmm_comparison])
display(hmm_comparison_df.round(4))
if hmm_comparison.get("Status", "").startswith("Fitted"):
    display(pd.DataFrame(
        hmm_model.transmat_,
        index=[f"State {i}" for i in range(hmm_model.n_components)],
        columns=[f"State {i}" for i in range(hmm_model.n_components)]
    ).round(3))

all_seven_overview = pd.DataFrame([
    ["Logistic Regression", "Classification", "CV/test ROC-AUC, recall, PR-AUC"],
    ["SVM (RBF)", "Classification", "CV/test ROC-AUC, recall, PR-AUC"],
    ["Random Forest", "Classification", "CV/test ROC-AUC, recall, PR-AUC"],
    ["XGBoost", "Classification", "CV/test ROC-AUC, recall, PR-AUC"],
    ["K-Means", "Clustering", "Silhouette, Davies-Bouldin, Calinski-Harabasz"],
    ["GMM (EM)", "Clustering", "Silhouette, Davies-Bouldin, Calinski-Harabasz, BIC"],
    ["HMM", "Sequential modelling", "Log-likelihood and transition probabilities"]
], columns=["Model", "Task", "Evaluation measures"])
print("4. Overview of all seven project models")
display(all_seven_overview)

best_model_name = classifier_comparison.iloc[0]["Model"]
best_model = fitted_pipelines[best_model_name]
best_predictions = best_model.predict(X_test)
best_probabilities = best_model.predict_proba(X_test)[:, 1]
tn, fp, fn, tp = confusion_matrix(y_test, best_predictions, labels=[0, 1]).ravel()

best_summary = pd.DataFrame([{
    "Best progression classifier": best_model_name,
    "CV ROC-AUC mean": classifier_comparison.iloc[0]["CV ROC-AUC"],
    "CV ROC-AUC std": classifier_comparison.iloc[0]["CV ROC-AUC std"],
    "Test accuracy": accuracy_score(y_test, best_predictions),
    "Test precision": precision_score(y_test, best_predictions, zero_division=0),
    "Test recall": recall_score(y_test, best_predictions, zero_division=0),
    "Test specificity": tn / max(1, tn + fp),
    "Test F1": f1_score(y_test, best_predictions, zero_division=0),
    "Test ROC-AUC": roc_auc_score(y_test, best_probabilities),
    "Test PR-AUC": average_precision_score(y_test, best_probabilities)
}])
print("5. Best predictive classifier")
display(best_summary.round(4))

ConfusionMatrixDisplay.from_predictions(
    y_test, best_predictions,
    display_labels=["No progression", "Progressed"],
    cmap="Blues", values_format="d"
)
plt.title(f"Final confusion matrix – {best_model_name}")
plt.tight_layout()
save_figure("23_best_model_confusion_matrix.png")
plt.show()

import joblib
joblib.dump(best_model, "PrediCare_Best_Progression_Classifier.joblib")
classifier_comparison.to_csv("PrediCare_Classifier_Comparison.csv", index=False)
cluster_quality_summary.to_csv("PrediCare_Clustering_Comparison.csv", index=False)
hmm_comparison_df.to_csv("PrediCare_HMM_Comparison.csv", index=False)
all_seven_overview.to_csv("PrediCare_All_Seven_Models_Overview.csv", index=False)
best_summary.to_csv("PrediCare_Best_Classifier_Results.csv", index=False)

print("Saved the best classifier, comparison tables and figures in:", FIGURE_DIR)
print(
    f"{best_model_name} was selected using the highest mean five-fold "
    f"cross-validation ROC-AUC ({best_summary.loc[0, 'CV ROC-AUC mean']:.3f}). "
    f"Its held-out test ROC-AUC was {best_summary.loc[0, 'Test ROC-AUC']:.3f}, "
    f"recall was {best_summary.loc[0, 'Test recall']:.3f}, and F1 was "
    f"{best_summary.loc[0, 'Test F1']:.3f}. External validation is needed "
    "before any clinical use."
)

### Exported outputs

All generated plots are saved as high-resolution PNG files in `PrediCare_Figures`. The comparison CSV files and selected classifier are saved in the Colab working directory.